# ***Part 1 — Python & Data Foundations: ingest, validate, clean, enrich***


In [ ]:
#Task 1.1 — Read the file the foundations way
import csv
#importing csv module
import json
#function parse_row(row) that converts age, fee_inr and prior_no_shows to int and distance_km to float, returning the converted dictionary
def parse_row(row):
  """Converting numeric fields to int/float; raises ValueError on blank or non-numeric strings."""
  converted = dict(row)
  converted["age"] = int(row["age"])
  converted["fee_inr"] = int(row["fee_inr"])
  converted["prior_no_shows"] = int(row["prior_no_shows"])
  converted["distance_km"] = float(row["distance_km"])
  return converted
accepted = []
rejected = []
#opening appointment file through with open()
# Load raw CSV from the local working directory so the notebook runs anywhere without hardcoded /content/ paths
with open("/content/appointments.csv", mode="r", newline="", encoding="utf-8") as f:
  #read csv file through csv.dictreader
  reader = csv.DictReader(f)
  headers = reader.fieldnames
  #print header count
  print("Headers:", headers)
  #print row count
  #looping over every row
  for row in reader:
    try:
      #try for accepted list
      clean_row = parse_row(row)
      accepted.append(clean_row)
    #except ValueError to put rows that fail into a rejected list
    except ValueError:
      rejected.append(row)
total_rows = len(accepted) + len(rejected)
print(f"Total rows read: {total_rows}")
print(f"Accepted rows: {len(accepted)}")
print(f"Rejected rows: {len(rejected)}")
#Saving rejected with json.dump(..., indent=2) as rejected_rows.json
with open("rejected_rows.json", mode="w", encoding="utf-8") as out_f:
  json.dump(rejected, out_f, indent=2)
print("Saved rejected_rows.json successfully.")

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020
Accepted rows: 988
Rejected rows: 32
Saved rejected_rows.json successfully.


### Task 1.1 — Load & Validate Appointment Data
Reads `appointments.csv` using Python's built-in `csv.DictReader` and enforces schema types via `parse_row()` (`int` for `age`, `fee_inr`, `prior_no_shows`; `float` for `distance_km`).

* **Valid rows** are parsed and stored in `accepted`.
* **Malformed rows** (`ValueError`/`TypeError`) are captured in `rejected` and exported to `rejected_rows.json` for auditing.
*

In [ ]:
#Task 1.2 — Inspect and clean, pass 1
import pandas as pd
# Start fresh from the raw CSV export (separate from Task 1.1's DictReader lists)
#Loading the file with pd.read_csv
df = pd.read_csv("appointments.csv")
#Print shape, info(), isnull().sum() and duplicated().sum()
raw_count = len(df)
print("Raw shape:", df.shape)
print("\nDataFrame info:",df.info())
print("\nAll Missing values per column:\n", df.isnull().sum())
print("\nExact number of duplicate rows:", df.duplicated().sum())
print("Distinct booking_channel values (raw):", df["booking_channel"].nunique())

# (a)exact duplicate rows first before cleaning text labels
df = df.drop_duplicates().copy()
after_dedup = len(df)

# (b)clean booking_channel with .str.strip().str.title()
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Distinct booking_channel values (clean):", df["booking_channel"].nunique(), df["booking_channel"].unique())

# (c)convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

# (d)drop rows where appt_date is missing after conversion or status is missing
df = df.dropna(subset=["appt_date", "status"]).copy()
#printing row count after each step
print(f"\nRow count progression: {raw_count} (raw) -> {after_dedup} (deduped) -> {len(df)} (valid dates/status)")

Raw shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

DataFrame info: None

All Missing values per column:
 appointment_id 

## Task 1.2 — Inspect & Clean (Pass 1)
- Load `appointments.csv` via `pd.read_csv()` and inspect raw shape, data types, missing values, duplicate rows, and `booking_channel` categories.
- Drop exact duplicate rows, standardize `booking_channel` casing (`.str.strip().str.title()`), parse `appt_date` and `booked_on` with `pd.to_datetime()`, and drop rows missing valid `appt_date` or `status`.


In [ ]:
#Task 1.3 — Clean, pass 2
import numpy as np

# (a) Treat any age below 1 or above 100 as invalid and set it to missing
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print(f"Invalid age rows flagged: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, "age"] = np.nan
print(f"Total missing ages before imputation: {df['age'].isnull().sum()}")

# (b) Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median.
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print(f"Imputing medians -> age: {age_median}, distance_km: {dist_median}")

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) compute Q1, Q3 and IQR with .quantile, and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip
for col in ["distance_km", "fee_inr"]:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  lower_fence = q1 - 1.5 * iqr
  upper_fence = q3 + 1.5 * iqr

  clipped_vals = np.clip(df[col], lower_fence, upper_fence)
  capped_count = (df[col] != clipped_vals).sum()
  df[col] = clipped_vals

  print(
      f"{col}: Q1={q1}, Q3={q3}, upper_fence={upper_fence:.4f} | values"
      f" capped={capped_count}"
  )

# (d)  Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int))
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
#values capped in each column
print(f"Largest fee_inr after capping and rounding: {df['fee_inr'].max()}")

Invalid age rows flagged: 5
Total missing ages before imputation: 17
Imputing medians -> age: 48.0, distance_km: 8.25
distance_km: Q1=4.625, Q3=13.0, upper_fence=25.5625 | values capped=26
fee_inr: Q1=500.0, Q3=787.5, upper_fence=1218.7500 | values capped=3
Largest fee_inr after capping and rounding: 1219


## Task 1.3 — Clean (Pass 2: Imputation, Outlier Capping & Rounding)
- Flag invalid `age` entries (`< 1` or `> 100`) as `NaN` and impute missing `age` (rounded integer) and `distance_km` values using column medians.
- Cap outliers in `distance_km` and `fee_inr` at `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` via `np.clip()`, then round `distance_km` to 1 decimal place and `fee_inr` to whole integers.
-

In [ ]:
#Task 1.4 — Features and a NumPy summary
#Create lead_days = days between booked_on and appt_date (use .dt.days) and no_show = 1 if status is "No-Show" else 0
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(
    f"No-show count: {df['no_show'].sum()} out of {len(df)}"
    f" ({df['no_show'].mean() * 100:.1f}%)"
)
# take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy() and, without loops, compute
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
#(a)the mean of each column (axis=0)
col_means = np.mean(X, axis=0)
print(
    "Column means [age, distance_km, fee_inr, lead_days]:"
    f" {np.round(col_means, 2)}"
)
#(b)the number and share of rows with lead_days > 14 using a boolean mask (mask.sum() counts the True values and mask.mean() gives the share);
lead_mask = X[:, 3] > 14
print(
    f"Rows with lead_days > 14: {lead_mask.sum()} ({lead_mask.mean() * 100:.1f}%)"
)
#(c)z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2
dist_col = X[:, 1]
z_scores = (dist_col - np.mean(dist_col)) / np.std(dist_col)
extreme_z_count = (np.abs(z_scores) > 2).sum()
print(f"Rows with |z| > 2 for distance_km: {extreme_z_count}")

lead_days range: 1 to 45
No-show count: 218 out of 990 (22.0%)
Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: 296 (29.9%)
Rows with |z| > 2 for distance_km: 53


## Task 1.4 — Feature Engineering & NumPy Summary
- Engineer `lead_days` (`(appt_date - booked_on).dt.days`) and binary `no_show` (`1` for `No-Show`, `0` otherwise), checking overall ranges and no-show rates.
- Convert `["age", "distance_km", "fee_inr", "lead_days"]` into a NumPy matrix to compute vectorized column means, the share of appointments with `lead_days > 14`, and `distance_km` z-scores (`|z| > 2`).


In [ ]:
#Task 1.5 — Enrich the clinics from a live API
import requests
#call the free Open-Meteo API with requests.get("https://api.open-meteo.com/v1/forecast",
#params={"latitude": ..., "longitude": ..., "hourly": "temperature_2m", "forecast_days": 1})
clinics_df = pd.read_csv("/content/clinics.csv")
url = "https://api.open-meteo.com/v1/forecast"
temps = []

for _, row in clinics_df.iterrows():
  clinic_name = row["clinic_name"]
  params = {
      "latitude": row["latitude"],
      "longitude": row["longitude"],
      "hourly": "temperature_2m",
      "forecast_days": 1,
  }

  temp_c = None
  fail_reason = ""

  # Attempt up to 2 times (initial call + 1 retry on RequestException)
  for attempt in range(2):
    try:
      resp = requests.get(url, params=params, timeout=10)
      #Checking status_code == 200
      if resp.status_code == 200:
        #reading the JSON
        data = resp.json()
        #first value of ["hourly"]["temperature_2m"] as temp_c
        #save clinics_enriched.csv (the original columns plus temp_c
        #df.to_csv("file.csv", index=False) writes a DataFrame to CSV)
        if (
            "hourly" in data
            and "temperature_2m" in data["hourly"]
            and len(data["hourly"]["temperature_2m"]) > 0
        ):
        #If the API fails: pass timeout=10 to requests.get
          temp_c = data["hourly"]["temperature_2m"][0]
          fail_reason = ""
          break
        else:
          fail_reason = "Missing hourly temperature_2m data in JSON"
      else:
        fail_reason = f"HTTP status {resp.status_code}"
    #wraping the call in try/except requests.exceptions.RequestException
    except requests.exceptions.RequestException as e:
      fail_reason = f"RequestException: {e}"
 # retrying that clinic once
 #if it still fails (or the status is not 200, or hourly is missing) store None as temp_c
  if temp_c is not None:
    print(f"[{clinic_name}] Status 200 -> temp_c = {temp_c}")
  else:
    print(f"[{clinic_name}] Failed after retry ({fail_reason}) -> temp_c = None")

  temps.append(temp_c)
# print the clinic's name and the reason
clinics_df["temp_c"] = temps
clinics_df.to_csv("clinics_enriched.csv", index=False)

print("\nSaved clinics_enriched.csv:")
print(clinics_df)

[Kothrud Family Clinic] Status 200 -> temp_c = 19.1
[Hadapsar Care Centre] Status 200 -> temp_c = 19.6
[Sitabuldi Health Point] Status 200 -> temp_c = 18.9
[Malviya Nagar Clinic] Status 200 -> temp_c = 14.7
[Vaishali Nagar Clinic] Status 200 -> temp_c = 14.7
[Edappally Medical Centre] Status 200 -> temp_c = 27.1
[Vijay Nagar Clinic] Status 200 -> temp_c = 17.1
[Gomti Nagar Care Point] Status 200 -> temp_c = 14.4

Saved clinics_enriched.csv:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6 

## Task 1.5 — Enrich Clinics via Open-Meteo API
- Query `https://api.open-meteo.com/v1/forecast` (`hourly="temperature_2m"`, `forecast_days=1`) for each clinic's `latitude` and `longitude` with `timeout=10` and a 1-retry fallback on `RequestException` or HTTP errors.
- Extract the first hourly temperature reading as `temp_c` (defaulting to `None` if unavailable) and export the enriched table to `clinics_enriched.csv`.
-

In [ ]:
#Task 1.6 — Export the clean table and write the SQL load script
# Sort by appointment_id and format dates as YYYY-MM-DD strings
df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# Keep exact 16 columns in the specified order
final_cols = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show",
]
df_clean = df[final_cols].copy()

# Verify uniqueness and zero nulls before exporting
unique_ids = df_clean["appointment_id"].nunique()
total_nulls = df_clean.isnull().sum().sum()
print(f"Distinct appointment_ids: {unique_ids} (expected 990)")
print(f"Total missing values across 16 columns: {total_nulls} (expected 0)")

assert unique_ids == 990, "appointment_id is not unique!"
assert total_nulls == 0, "Missing values found in final table!"

# Save appointments_clean.csv
df_clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv successfully.")

# Generate load_data.sql using for loops and f-strings
# Write clinics statements FIRST because appointments.clinic_id references clinics(clinic_id)
with open("load_data.sql", mode="w", encoding="utf-8") as sql_file:
  for _, row in clinics_df.iterrows():
    temp_val = "NULL" if pd.isna(row["temp_c"]) else f"{row['temp_c']}"
    # Support either 'clinic_name' or 'name' column from clinics_enriched.csv
    c_name = row["clinic_name"] if "clinic_name" in row else row["name"]
    c_id = row["clinic_id"] if "clinic_id" in row else row["id"]
    stmt = (
        f"INSERT INTO clinics VALUES ('{c_id}', '{c_name}', '{row['city']}', "
        f"'{row['tier']}', {row['latitude']}, {row['longitude']}, {temp_val});\n"
    )
    sql_file.write(stmt)

  for _, row in df_clean.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ('{row['appointment_id']}',"
        f" '{row['patient_id']}', '{row['clinic_id']}', '{row['booked_on']}',"
        f" '{row['appt_date']}', {row['age']}, '{row['gender']}',"
        f" {row['distance_km']}, {row['fee_inr']}, '{row['booking_channel']}',"
        f" '{row['visit_type']}', '{row['reminder_sent']}',"
        f" {row['prior_no_shows']}, '{row['status']}', {row['lead_days']},"
        f" {row['no_show']});\n"
    )
    sql_file.write(stmt)

# Verify total INSERT statement count
with open("load_data.sql", mode="r", encoding="utf-8") as f:
  line_count = sum(1 for line in f if line.strip().startswith("INSERT INTO"))

print(
    f"Saved load_data.sql with {line_count} INSERT statements (8 clinics + 990"
    " appointments)."
)

Distinct appointment_ids: 990 (expected 990)
Total missing values across 16 columns: 0 (expected 0)
Saved appointments_clean.csv successfully.
Saved load_data.sql with 998 INSERT statements (8 clinics + 990 appointments).


## Task 1.6 — Export Clean Table & Generate SQL Seed Script
- Sort records by `appointment_id`, format dates as `YYYY-MM-DD` strings, enforce the exact 16-column order, and assert 990 unique IDs with zero nulls before saving `appointments_clean.csv`.
- Generate `load_data.sql` with 998 `INSERT` statements (8 `clinics` rows first, followed by 990 `appointments` rows to satisfy foreign-key constraints) and verify the line count.
-